<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_04_Statistics_and_Probability/Week_4_Regression_and_Modeling/Day_28_Month_4_Final_Review_and_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 28: Month 4 Final Review and Capstone Project

## Phase 1 | Month 4 | Week 4 | Day 28

**🎯 Goal:** Demonstrate mastery of the entire Month 4 curriculum through a comprehensive statistical analysis.
**⏱️ Study Session:** ~3 hours (capstone)

---

### Month 4 Topics Mastered

**Week 1 – Descriptive Statistics**
1. ✓ Introduction to Statistics — data types, population vs sample
2. ✓ Measures of Central Tendency — mean, median, mode
3. ✓ Measures of Variability — variance, std, IQR, CV
4. ✓ Distributions and Frequency Tables
5. ✓ Percentiles, Quartiles, Box Plots
6. ✓ Skewness and Kurtosis

**Week 2 – Probability Theory**
7. ✓ Probability Rules and Simulation
8. ✓ Conditional Probability and Bayes' Theorem
9. ✓ Random Variables, PMF, PDF, CDF
10. ✓ Binomial and Poisson Distributions
11. ✓ Normal Distribution and Z-scores
12. ✓ Sampling Distributions and CLT

**Week 3 – Inferential Statistics**
13. ✓ Statistical Inference and Estimation
14. ✓ Confidence Intervals
15. ✓ Hypothesis Testing Fundamentals
16. ✓ t-Tests
17. ✓ Chi-Square and ANOVA
18. ✓ Correlation Analysis

**Week 4 – Regression and Modeling**
19. ✓ Simple Linear Regression
20. ✓ Multiple Linear Regression
21. ✓ Regression Diagnostics
22. ✓ Logistic Regression
23. ✓ Model Evaluation
24. ✓ statsmodels Formula API

## 🏆 Capstone Project: Kenya Financial Inclusion Analysis

You will perform a **full statistical analysis** on a simulated Kenya Financial Inclusion dataset — the type of analysis published annually by FSD Kenya and CBK.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(2024)
N = 3000

df = pd.DataFrame({
    'person_id':      range(1, N+1),
    'age':            np.random.normal(32, 12, N).clip(18, 80).astype(int),
    'gender':         np.random.choice(['Male','Female'], N, p=[0.52,0.48]),
    'county':         np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret',
                                         'Machakos','Kakamega','Kisii','Meru','Thika'], N),
    'urban':          np.random.binomial(1, 0.55, N),
    'monthly_income': np.random.lognormal(9.5, 1.0, N).round(0),
    'years_education':np.random.choice([6,9,12,16], N, p=[0.15,0.25,0.40,0.20]),
    'has_bank_acct':  None,
    'uses_mobile_money': None,
    'has_insurance':  None,
})

# Simulate financial inclusion outcomes
logit_bank = (-3
              + 0.8 * df['urban']
              + 0.15 * df['years_education']
              + 0.00002 * df['monthly_income'])
df['has_bank_acct'] = (np.random.rand(N) < 1/(1+np.exp(-logit_bank))).astype(int)

logit_mm = (-1
            + 0.5 * df['urban']
            + 0.08 * df['years_education']
            + 0.00001 * df['monthly_income'])
df['uses_mobile_money'] = (np.random.rand(N) < 1/(1+np.exp(-logit_mm))).astype(int)
df['has_insurance'] = (np.random.rand(N) < 0.12).astype(int)

print(df.describe().round(1))
print(f'\nBank account rate:   {df.has_bank_acct.mean():.3f}')
print(f'Mobile money rate:   {df.uses_mobile_money.mean():.3f}')
print(f'Insurance rate:      {df.has_insurance.mean():.3f}')

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(2024)
N = 3000
df = pd.DataFrame({
    'age':            np.random.normal(32,12,N).clip(18,80).astype(int),
    'gender':         np.random.choice(['Male','Female'],N,p=[0.52,0.48]),
    'county':         np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'],N),
    'urban':          np.random.binomial(1,0.55,N),
    'monthly_income': np.random.lognormal(9.5,1.0,N).round(0),
    'years_education':np.random.choice([6,9,12,16],N,p=[0.15,0.25,0.40,0.20]),
})
logit_bank = -3 + 0.8*df['urban'] + 0.15*df['years_education'] + 0.00002*df['monthly_income']
df['has_bank_acct'] = (np.random.rand(N) < 1/(1+np.exp(-logit_bank))).astype(int)
df['uses_mobile_money'] = (np.random.rand(N) < 1/(1+np.exp(-1+0.5*df['urban']+0.08*df['years_education']))).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1 – Income distribution
axes[0,0].hist(df['monthly_income'], bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[0,0].set_title('Monthly Income Distribution', fontweight='bold')
axes[0,0].set_xlabel('Income (KES)')

# 2 – Bank account rate by education
bank_edu = df.groupby('years_education')['has_bank_acct'].mean()
axes[0,1].bar(bank_edu.index.astype(str), bank_edu.values*100, color='tomato')
axes[0,1].set_title('Bank Account Rate by Education', fontweight='bold')
axes[0,1].set_ylabel('%')
axes[0,1].set_xlabel('Years of Education')

# 3 – Urban vs Rural inclusion rates
urban_rates  = df[df.urban==1][['has_bank_acct','uses_mobile_money']].mean()
rural_rates  = df[df.urban==0][['has_bank_acct','uses_mobile_money']].mean()
x = np.arange(2); w = 0.3
axes[0,2].bar(x-w/2, urban_rates.values*100, w, label='Urban', color='navy', alpha=0.8)
axes[0,2].bar(x+w/2, rural_rates.values*100, w, label='Rural', color='orange', alpha=0.8)
axes[0,2].set_xticks(x); axes[0,2].set_xticklabels(['Bank Account','Mobile Money'])
axes[0,2].set_title('Financial Inclusion: Urban vs Rural', fontweight='bold')
axes[0,2].set_ylabel('%'); axes[0,2].legend()

# 4 – Age distribution
axes[1,0].hist(df['age'], bins=30, color='green', alpha=0.7, edgecolor='white')
axes[1,0].set_title('Age Distribution', fontweight='bold')
axes[1,0].set_xlabel('Age (years)')

# 5 – Bank rate by county
county_rate = df.groupby('county')['has_bank_acct'].mean().sort_values(ascending=False)
axes[1,1].barh(county_rate.index, county_rate.values*100, color='purple', alpha=0.7)
axes[1,1].set_title('Bank Account Rate by County', fontweight='bold')
axes[1,1].set_xlabel('%')

# 6 – Income by bank account status
for status, color, label in [(0,'orange','No Bank Acct'),(1,'steelblue','Has Bank Acct')]:
    d = df[df.has_bank_acct==status]['monthly_income']
    axes[1,2].hist(d, bins=50, alpha=0.5, color=color, label=label)
axes[1,2].set_title('Income by Bank Account Status', fontweight='bold')
axes[1,2].legend()

plt.tight_layout()
fig.savefig('/tmp/day28_capstone.png', dpi=120)
print('Capstone dashboard saved')
plt.close()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

np.random.seed(2024)
N = 3000
df = pd.DataFrame({
    'age':            np.random.normal(32,12,N).clip(18,80).astype(int),
    'gender':         np.random.choice(['Male','Female'],N,p=[0.52,0.48]),
    'urban':          np.random.binomial(1,0.55,N),
    'monthly_income': np.random.lognormal(9.5,1.0,N).round(0),
    'years_education':np.random.choice([6,9,12,16],N,p=[0.15,0.25,0.40,0.20]),
})
logit_bank = -3+0.8*df['urban']+0.15*df['years_education']+0.00002*df['monthly_income']
df['has_bank_acct'] = (np.random.rand(N) < 1/(1+np.exp(-logit_bank))).astype(int)

# 1. Two-sample t-test: does income differ by bank account ownership?
group0 = df[df.has_bank_acct==0]['monthly_income']
group1 = df[df.has_bank_acct==1]['monthly_income']
t, p = stats.ttest_ind(group0, group1, equal_var=False)
print(f'=== t-Test: Income by Bank Account ===')
print(f'Mean income (no account): KES {group0.mean():,.0f}')
print(f'Mean income (has account): KES {group1.mean():,.0f}')
print(f't={t:.4f},  p={p:.4e}')

# 2. Chi-square: independence of urban status and bank ownership
contingency = pd.crosstab(df['urban'], df['has_bank_acct'])
chi2, p_chi, dof, _ = stats.chi2_contingency(contingency)
print(f'\n=== Chi-Square: Urban vs Bank Account ===')
print(f'Chi2={chi2:.4f},  p={p_chi:.4e},  significant: {p_chi < 0.05}')

# 3. Logistic regression model
model = smf.logit('has_bank_acct ~ age + C(gender) + urban + monthly_income + years_education', data=df).fit(disp=0)
print(f'\n=== Logistic Regression AUC ===')
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df['has_bank_acct'], model.predict())
print(f'AUC = {auc:.4f}')

## 📋 Month 4 Complete — Statistics and Probability

### Key Takeaways

| Area | What You Can Now Do |
|------|--------------------|
| **Descriptive Stats** | Summarise any dataset with mean, std, IQR, skewness, box plots |
| **Probability** | Apply Bayes' theorem, work with Binomial/Poisson/Normal |
| **Inference** | Build CIs, run t-tests, chi-square, ANOVA |
| **Regression** | Fit, diagnose, and evaluate linear and logistic models |

### 🚀 What Comes Next
**Phase 2, Month 5: Data Cleaning and Preprocessing** — prepare messy real-world data for analysis.

### 🌙 Capstone Challenge
Download the FSD Kenya FinAccess 2021 survey data from [fsdkenya.org](https://fsdkenya.org) and replicate your analysis on real data. Share your findings as a one-page statistical report.